# ARE-002.2 - YummyAnime API Exploration
One public profile, up to four GET requests, public X-Application only. No Bearer, cookies,
retries, redirects, database, or production client.

Install: `python -m pip install -e ".[research]"`. Select the local .venv kernel.
Set YUMMYANIME_APPLICATION_TOKEN in the root .env. Run all cells in order.
On an API error no further requests are sent; empty tables do not mean an empty profile.

PRIVACY: profile and table outputs contain personal data during your session.
Jupyter may autosave them. Clear Outputs of All Cells before saving or sharing.
This repository copy has no saved outputs. Never export the DataFrame.

Official documentation: https://ru.yummyani.me/api/swagger
API: https://api.yani.tv. Website counters are entered manually; no site scraping.


In [ ]:
import os
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

root = Path.cwd().resolve()
if not (root / "pyproject.toml").is_file():
    root = root.parent
if not (root / "pyproject.toml").is_file():
    raise RuntimeError("Run from the project root or notebooks directory.")
load_dotenv(root / ".env", override=False)
token = os.getenv("YUMMYANIME_APPLICATION_TOKEN", "").strip()
nickname = os.getenv("YUMMYANIME_TEST_NICKNAME", "").strip() or input("Enter your nickname: ") or "First_Light"
base_url = "https://api.yani.tv"
http_log = []
limitations = []
api_ready = bool(token)
if not api_ready:
    limitations.append("Public application token missing; no requests sent.")


def get_response(path, label):
    """No retries, redirects, cookies, body, URL, or token logging."""
    global api_ready
    if not api_ready:
        return None
    try:
        response = requests.get(
            base_url + path,
            headers={"X-Application": token, "Accept": "application/json"},
            timeout=(10, 30), allow_redirects=False,
        )
    except requests.RequestException:
        http_log.append({"endpoint": label, "HTTP": None, "result": "network error"})
        limitations.append("Network error; stopped; details omitted.")
        api_ready = False
        return None
    http_log.append({"endpoint": label, "HTTP": response.status_code,
                     "result": "OK" if response.status_code == 200 else "stopped"})
    if response.status_code != 200:
        limitations.append(f"{label}: HTTP {response.status_code}; stopped.")
        api_ready = False
        return None
    try:
        payload = response.json()
    except ValueError:
        payload = None
    if (not isinstance(payload, dict) or "response" not in payload
            or payload.get("error") or payload.get("errors")):
        limitations.append(f"{label}: invalid JSON envelope or API error; stopped.")
        api_ready = False
        return None
    return payload["response"]


## Profile
Display only ID, nickname, and exact spelling match.

In [ ]:
profile = get_response("/users/" + quote(nickname, safe=""), "GET /users/{nickname}")
user_id = profile.get("id") if isinstance(profile, dict) else None
profile_found = isinstance(user_id, int) and not isinstance(user_id, bool)
if profile_found:
    display(pd.DataFrame([{
        "user_id": user_id,
        "nickname": profile.get("nickname"),
        "exact_nickname_match": profile.get("nickname") == nickname,
    }]))
else:
    api_ready = False
    limitations.append("Profile not found or no integer user ID.")
display(pd.DataFrame(http_log))


## Lists and DataFrame
Use user.rating, never the global anime rating. Favorites may overlap watch statuses.
The documented date means list addition, not a history of status changes.


In [ ]:
rows = get_response(f"/users/{user_id}/lists", "GET /users/{id}/lists") if profile_found else None
if rows is None:
    rows = []
elif not isinstance(rows, list) or any(not isinstance(row, dict) for row in rows):
    limitations.append("Unexpected list schema; DataFrame remains empty.")
    rows = []
    api_ready = False


def nested(row, *keys):
    value = row
    for key in keys:
        if not isinstance(value, dict):
            return None
        value = value.get(key)
    return value


columns = ["anime_id", "title", "user_rating", "list_id", "status_alias", "is_fav", "date", "myanimelist_id"]
df = pd.DataFrame([{
    "anime_id": row.get("anime_id"),
    "title": row.get("title"),
    "user_rating": nested(row, "user", "rating"),
    "list_id": nested(row, "user", "list", "list", "id"),
    "status_alias": nested(row, "user", "list", "list", "href"),
    "is_fav": nested(row, "user", "list", "is_fav"),
    "date": row.get("date"),
    "myanimelist_id": nested(row, "remote_ids", "myanimelist_id"),
} for row in rows], columns=columns)
for col in ["anime_id", "list_id", "date", "myanimelist_id"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
df["user_rating"] = pd.to_numeric(df["user_rating"], errors="coerce").astype("Float64")
df["is_fav"] = df["is_fav"].map(lambda value: value if isinstance(value, bool) else pd.NA).astype("boolean")
display(pd.DataFrame(http_log))
display(df.head(10))
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum()}))
display(pd.Series({"records": len(df), "unique_anime_id": df["anime_id"].nunique(),
                   "duplicate_anime_id": int(df["anime_id"].dropna().duplicated().sum())}, name="summary"))


## Watch statuses and original response shape
Official schema: https://ru.yummyani.me/api/swagger/swagger-ui-init.js
Nested watch IDs: 0 Watching, 1 Planned, 2 Completed, 3 Dropped, 5 On hold.
The endpoint accepts 0..5; ID 4 is the favorites filter, not a watch category
(verified by comparing membership, not by inventing watch IDs).
Favorites and watch statuses overlap. Null nested list is preserved as missing.
Do not replace null with 4 or infer watched/planned from favorites.


In [ ]:
status_names = {0: "Watching", 1: "Planned", 2: "Completed", 3: "Dropped", 5: "On hold"}
status_labels = df["list_id"].map(status_names).astype("string")
unknown = df["list_id"].notna() & status_labels.isna()
status_labels.loc[unknown] = df.loc[unknown, "list_id"].map(lambda value: f"Other ({value})")
status_labels = status_labels.fillna("No status")
counts = status_labels.value_counts()
status_table = counts.reindex(list(status_names.values()) + [label for label in counts.index if label not in status_names.values()], fill_value=0).rename_axis("Status").to_frame("API")
display(status_table)

# Inspect only structural shapes, not titles, profile IDs, or anime IDs.
missing_status_rows = [row for row in rows if nested(row, "user", "list", "list", "id") is None]
status_shapes = pd.DataFrame([{
    "user_type": type(row.get("user")).__name__,
    "user_list_type": type(nested(row, "user", "list")).__name__,
    "nested_list_present": "list" in (nested(row, "user", "list") or {}),
    "nested_list_type": type(nested(row, "user", "list", "list")).__name__,
    "raw_list_id": nested(row, "user", "list", "list", "id"),
    "is_fav": nested(row, "user", "list", "is_fav"),
} for row in missing_status_rows])
display(status_shapes)
raw_status_pairs = pd.DataFrame([{
    "raw_id": nested(row, "user", "list", "list", "id"),
    "raw_alias": nested(row, "user", "list", "list", "href"),
} for row in rows], columns=["raw_id", "raw_alias"])
display(raw_status_pairs.value_counts(dropna=False).rename("count").to_frame())


## Ratings, favorites, MAL IDs
Use only user.rating; preserve zero values separately from null.
The schema says "Current user rating", which does not unambiguously identify
whose rating is returned for another profile. Cross-check against the requested
profile's rating statistics below. Matching distributions support ownership but
do not prove each per-item value or that zero always means unrated.


In [ ]:
rated = df.loc[df["user_rating"].gt(0).fillna(False), "user_rating"]
mal_present = df["myanimelist_id"].gt(0).fillna(False)
rating_summary = pd.Series({
    "rated_count": len(rated),
    "rating_min": rated.min() if len(rated) else pd.NA,
    "rating_max": rated.max() if len(rated) else pd.NA,
    "zero_ratings": int(df["user_rating"].eq(0).sum()),
    "missing_ratings": int(df["user_rating"].isna().sum()),
    "favorites": int(df["is_fav"].sum()),
    "mal_id_count": int(mal_present.sum()),
    "mal_id_share": float(mal_present.mean()) if len(df) else pd.NA,
}, name="metric")
display(rating_summary)
display(rated.value_counts().sort_index().rename_axis("User rating").to_frame("Count"))


## Manual website comparison
No website parsing. Replace None counters manually, then rerun this cell.
Favorites overlap watch categories and must not be added to them.
Restore None values and clear outputs before saving for the repository.
Matching counts do not verify every item's rating or prove universal API completeness.


In [ ]:
website_counts = {"Completed": None, "Planned": None, "Watching": None,
                  "Dropped": None, "On hold": None, "Favorites": None}
api_counts = {label: int(counts.get(label, 0)) for label in website_counts}
api_counts["Favorites"] = int(df["is_fav"].sum())
comparison = pd.DataFrame({
    "API": api_counts,
    "Website": pd.Series(website_counts, dtype="Int64"),
})
comparison["Match"] = comparison["API"].astype("Int64").eq(comparison["Website"])
if not api_ready:
    comparison["Match"] = pd.NA
    limitations.append("API analysis incomplete: empty tables do not prove an empty profile.")
display(comparison)


## Independent API cross-checks (one profile only)
Two additional read-only requests: favorites filter and rating distribution.
Compare sets of anime IDs, not just counts. Missing/null IDs are reported separately.
No retries or alternate authorization after any API failure.


In [ ]:
favorite_response = get_response(f"/users/{user_id}/lists/4", "GET /users/{id}/lists/4") if profile_found else None
favorite_check = None
if isinstance(favorite_response, list) and all(isinstance(r, dict) for r in favorite_response):
    expected_fav = set(df.loc[df["is_fav"].fillna(False), "anime_id"].dropna())
    endpoint_fav = {r.get("anime_id") for r in favorite_response if isinstance(r.get("anime_id"), int)}
    favorite_check = expected_fav == endpoint_fav
    display(pd.Series({"favorite_records": len(favorite_response),
                       "unique_favorite_ids": len(endpoint_fav),
                       "favorite_membership_matches": favorite_check}))
elif api_ready:
    limitations.append("Unexpected favorites response; stop further API requests.")
    api_ready = False

rating_response = get_response(f"/users/{user_id}/stats/ratings", "GET /users/{id}/stats/ratings") if profile_found else None
rating_distribution_matches = None
if isinstance(rating_response, list) and all(isinstance(r, dict) for r in rating_response):
    stats = pd.DataFrame(rating_response, columns=["rating", "count"])
    stats["rating"] = pd.to_numeric(stats["rating"], errors="coerce")
    stats["count"] = pd.to_numeric(stats["count"], errors="coerce")
    valid_stats = stats.notna().all().all() and stats["count"].ge(0).all()
    if valid_stats:
        server_positive = stats.loc[stats["rating"].gt(0)].groupby("rating")["count"].sum()
        local_positive = rated.value_counts().sort_index()
        rating_comparison = pd.concat([local_positive.rename("List"), server_positive.rename("Profile stats")], axis=1).fillna(0)
        rating_distribution_matches = bool(rating_comparison["List"].eq(rating_comparison["Profile stats"]).all())
        display(rating_comparison)
        display(pd.Series({"positive_distributions_match": rating_distribution_matches,
                           "stats_zero_bucket": int(stats.loc[stats["rating"].eq(0), "count"].sum())}))
    else:
        limitations.append("Invalid rating statistics; ownership cross-check inconclusive.")
elif api_ready:
    limitations.append("Unexpected rating-statistics response.")
    api_ready = False


## Data-quality summary
Raw API objects remain unchanged in rows. Numeric parsing is for analysis only;
count values that could not be parsed, rather than silently treating them as missing.
No deduplication or imputation is performed. Negative/zero IDs are not valid mappings.


In [ ]:
raw_fields = {
    "anime_id": [r.get("anime_id") for r in rows],
    "list_id": [nested(r, "user", "list", "list", "id") for r in rows],
    "user_rating": [nested(r, "user", "rating") for r in rows],
    "myanimelist_id": [nested(r, "remote_ids", "myanimelist_id") for r in rows],
    "date": [r.get("date") for r in rows],
}
parse_failures = {}
for name, values in raw_fields.items():
    raw = pd.Series(values, dtype="object")
    parse_failures[name] = int((raw.notna() & pd.to_numeric(raw, errors="coerce").isna()).sum())
quality = pd.Series({
    "records": len(df),
    "unique_anime_ids": df["anime_id"].nunique(),
    "duplicate_anime_ids": int(df["anime_id"].dropna().duplicated().sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "missing_anime_ids": int(df["anime_id"].isna().sum()),
    "invalid_nonpositive_anime_ids": int(df["anime_id"].le(0).sum()),
    "missing_watch_status": int(df["list_id"].isna().sum()),
    "missing_status_and_favorite": int((df["list_id"].isna() & df["is_fav"].fillna(False)).sum()),
    "unknown_watch_status": int((df["list_id"].notna() & ~df["list_id"].isin(status_names)).sum()),
    "missing_ratings": int(df["user_rating"].isna().sum()),
    "negative_ratings": int(df["user_rating"].lt(0).sum()),
    "invalid_favorite_values": sum(not isinstance(nested(r, "user", "list", "is_fav"), bool) for r in rows),
    "valid_mal_ids": int(mal_present.sum()),
    "mal_id_share": float(mal_present.mean()) if len(df) else pd.NA,
    "duplicate_mal_ids": int(df.loc[mal_present, "myanimelist_id"].duplicated().sum()),
}, name="Value")
display(quality)
display(pd.Series(parse_failures, name="Numeric parse failures"))
print("Favorites set matches:", favorite_check)
print("Profile rating distribution matches:", rating_distribution_matches)
print("Zero means unrated: NOT independently confirmed; verify item values in UI.")
print("Per-item rating ownership: supported only if profile distribution matches; UI confirmation pending.")


## Conclusions and limits
ALS fold-in requires trained item factors, a verified MAL-to-training-item mapping,
and a policy for converting ratings, statuses, and favorites to confidence.
Planned items are not automatically positive watches. Favorites can have no watch status.
This API provides a list snapshot, not a complete event history.
Offline ALS training needs a separately licensed multi-user interaction dataset.


In [ ]:
display(pd.DataFrame(http_log))
print("API analysis completed:", bool(api_ready and profile_found))
print("Website completeness verified:", bool(api_ready and comparison["Website"].notna().all() and comparison["Match"].fillna(False).all()))
for limitation in dict.fromkeys(limitations):
    print(limitation)
print("DataFrames and responses remain in memory only. Clear all outputs before saving.")


In [ ]:
df